# 03 — SAI, Comparables, Outliers, Clustering & Draft StealsParts 3, 4, 5, 7, 8 in one pass: the custom Scouting Athletic Index, historical comparables, outlier/freak-athlete detection, draft steals vs. reaches, and position clustering.

In [ ]:
import sys; sys.path.insert(0, '../src')import pandas as pd, numpy as npimport matplotlib.pyplot as plt

## Scouting Athletic Index (SAI)

In [ ]:
df = pd.read_csv('../data/processed/combine_with_sai.csv')df[['Player','PosGroup','Year','SAI']].sort_values('SAI', ascending=False).head(10)

In [ ]:
df['SAI'].hist(bins=40, color='#3C6E47', figsize=(7,3.5))plt.title('SAI Distribution — all prospects'); plt.xlabel('SAI'); plt.show()

## Historical comparables (3 distance metrics)

In [ ]:
from comparables import find_comparablesresult, err = find_comparables(df, 'Travis Hunter', method='euclidean', k=5)pd.DataFrame(result['comparables'])[['Player','Year','School','SAI','similarity_pct']]

In [ ]:
for method in ['euclidean','cosine','mahalanobis']:    r, e = find_comparables(df, 'Calvin Johnson', method=method, k=3)    print(method, '->', [c['Player'] for c in r['comparables']])

## Outlier detection (Isolation Forest + LOF)

In [ ]:
from outliers import detect_outliersd = detect_outliers(df)freaks = d[(d['iso_pctile']>=85) & (d['SAI']>=90)].sort_values('SAI', ascending=False)freaks[['Player','PosGroup','Year','SAI','iso_pctile']].head(12)

## Draft steals & reaches

In [ ]:
preds = pd.read_csv('../data/processed/combine_with_predictions.csv')from draft_steals import compute_stealssteals_df = compute_steals(preds)steals_df.sort_values('ValueDelta', ascending=False)[['Player','PosGroup','Year','Pick','PredictedDraftValue','ValueDelta']].head(10)

**Caveat:** since the underlying model only explains ~20% of draft variance, predictions regress toward the mean, so early picks systematically look like 'reaches' and late picks like 'steals.' See `report.pdf` Part 7 for the full discussion.

## Position clustering into athletic archetypes

In [ ]:
from clustering import cluster_positionwr = cluster_position(df, 'WR')for name, info in wr['cluster_summary'].items():    print(f"{name:20s} n={info['n']:4d}  mean_40={info['mean_40']}  mean_SAI={info['mean_SAI']}")

In [ ]:
assign = pd.DataFrame(wr['player_assignments'])fig, ax = plt.subplots(figsize=(7,5.5))for name, grp in assign.groupby('ClusterName'):    ax.scatter(grp.PCA1, grp.PCA2, label=name, alpha=0.6, s=16)ax.legend(fontsize=8); ax.set_title('WR Archetypes (K-Means + PCA)'); plt.show()